# Predicting Low-Carbon Hours for Data-Center Computing

**Course:** AC209A  

**Canvas project group:** 14 

**Group members:** Minh-Thy Tyler, Gabe Gonzalez, Noah Schiro

## 1. Getting the Data

We obtained the PJM balancing authority Excel file from the EIA
Hourly Electric Grid Monitor.

Source: https://www.eia.gov/electricity/gridmonitor/knownissues/xls/PJM.xlsx

Download date: November 2 2026

The original file is stored locally at `data/raw/PJM.xlsx`.
We will inspect its sheets and date coverage before selecting our
proposed study period of January 2022 through December 2025.

In [1]:
# packages
from pathlib import Path
import pandas as pd

In [2]:
data_path = Path("../data/raw/PJM.xlsx")

#inspecing available sheets before choosing which ones to load 
excel_file = pd.ExcelFile(data_path)
print("Available sheets:", excel_file.sheet_names)

Available sheets: ['Published Hourly Data', 'Published Daily Data', 'BA Full Names', 'Hourly Charts', 'Daily Charts', 'Known Data Issues', 'Notes', 'Imbalance Thresholds', 'Range Thresholds']


In [3]:
#loading in hourly data
hourly_df = pd.read_excel(excel_file, sheet_name="Published Hourly Data")

In [4]:
#displaying first five rows
display(hourly_df.head())

#size of the entire downloaded Excel file
size_mb = data_path.stat().st_size / 1_000_000
print(f"Downloaded Excel file size: {size_mb:.2f} MB")

,BA,UTC time,Local date,Hour,Local time,Time zone,Generation only?,Demand forecast,Demand,Net generation,...,CO2 Emissions: OIL,CO2 Emissions: Other,CO2 Emissions Generated,CO2 Emissions Imported,CO2 Emissions Exported,CO2 Emissions Consumed,Positive Generation,Consumed Electricity,CO2 Emissions Intensity for Generated Electricity,CO2 Emissions Intensity for Consumed Electricity
0,PJM,2015-07-01 05:00:00,2015-07-01,1,2015-07-01 01:00:00,Eastern,N,29415.0,84024.0,83944.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PJM,2015-07-01 06:00:00,2015-07-01,2,2015-07-01 02:00:00,Eastern,N,27687.0,79791.0,80091.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PJM,2015-07-01 07:00:00,2015-07-01,3,2015-07-01 03:00:00,Eastern,N,26574.0,76760.0,77994.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PJM,2015-07-01 08:00:00,2015-07-01,4,2015-07-01 04:00:00,Eastern,N,26029.0,74931.0,76820.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,PJM,2015-07-01 09:00:00,2015-07-01,5,2015-07-01 05:00:00,Eastern,N,26220.0,74368.0,76268.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Downloaded Excel file size: 114.54 MB


## 2. Understanding the data

In [5]:
#Loading in known issues and notes about our data to have more of a comprehensive understanding
issues_df = pd.read_excel(excel_file, sheet_name = "Known Data Issues")
notes_df = pd.read_excel(excel_file, sheet_name = "Notes")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [6]:
display(issues_df.head())
display(notes_df.head())

,BA,UTC time,Local date,Hour,Data issue,Local time,Time zone,Generation only?,DF,D,...,File 2,File D,File 0 Processed Date,File 1 Processed Date,File 2 Processed Date,File D Processed Date,File 0 issue,File 1 issue,File 2 issue,File D issue
0,PJM,2015-07-01 05:00:00,2015-07-01,1,0,2015-07-01 01:00:00,Eastern,N,29415.0,84024.0,...,PJMdaily1842015,NaN,2016-02-20,2015-07-02,2015-07-03,NaT,0,0,0,0
1,PJM,2015-07-01 06:00:00,2015-07-01,2,0,2015-07-01 02:00:00,Eastern,N,27687.0,79791.0,...,PJMdaily1842015,NaN,2016-02-20,2015-07-02,2015-07-03,NaT,0,0,0,0
2,PJM,2015-07-01 07:00:00,2015-07-01,3,0,2015-07-01 03:00:00,Eastern,N,26574.0,76760.0,...,PJMdaily1842015,NaN,2016-02-20,2015-07-02,2015-07-03,NaT,0,0,0,0
3,PJM,2015-07-01 08:00:00,2015-07-01,4,0,2015-07-01 04:00:00,Eastern,N,26029.0,74931.0,...,PJMdaily1842015,NaN,2016-02-20,2015-07-02,2015-07-03,NaT,0,0,0,0
4,PJM,2015-07-01 09:00:00,2015-07-01,5,0,2015-07-01 05:00:00,Eastern,N,26220.0,74368.0,...,PJMdaily1842015,NaN,2016-02-20,2015-07-02,2015-07-03,NaT,0,0,0,0


,"The information submitted by reporting entities is preliminary data and is made available ""as-is"" by EIA. Neither EIA nor reporting entities are responsible for reliance on the data for any specific use.",Unnamed: 1
0,NaN,NaN
1,NaN,For more details see: About the EIA-930 data
2,NaN,NaN
3,Column,Description
4,BA,2-4 letter code that identifies the balancing ...


Looking at the data, each row in the 'Published Hourly Data' dataset represents one hourly observation for the PJM balancing authority. There are 108 columns, so we will not be discussing what each column represents individually, the relavanet columns that we will be using for our project describe the timestamp, electricity demand, generation, electricity exchanges, and estimated CO2 emissions and carbon intesnsity, along with any imputed and adjusted versions of the aformentioned measurements.

In [8]:
#number of rows and columns:
print("\nShape of table:")
print(f"Rows: {hourly_df.shape[0]:,}")
print(f"Columns: {hourly_df.shape[1]:,}")

#Column types and non-missing counts
print("\nHourly Data Info:")
hourly_df.info(show_counts=True)



Shape of table:
Rows: 99,409
Columns: 108

Hourly Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 99409 entries, 0 to 99408
Columns: 108 entries, BA to CO2 Emissions Intensity for Consumed Electricity
dtypes: datetime64[us](3), float64(101), int64(1), str(3)
memory usage: 81.9 MB


In [ ]:
# summary statistics of numerical columns
hourly_df.select_dtypes(include="number").describe().T

,count,mean,std,min,25%,50%,75%,max
Hour,99409.0,12.500236,6.922492e+00,1.000000,7.000000,13.000000,19.000000,2.500000e+01
Demand forecast,99025.0,90665.298712,1.695529e+04,21905.000000,79393.000000,88238.000000,99950.000000,1.655570e+05
Demand,99142.0,133606.816919,8.480803e+06,56260.000000,80951.000000,89356.500000,101172.000000,2.147480e+09
Net generation,99117.0,136451.370613,8.480797e+06,57919.000000,83496.000000,92124.000000,104118.000000,2.147480e+09
Total interchange,99118.0,2909.377086,3.126764e+03,-8156.000000,1199.000000,2978.000000,4605.000000,5.901060e+05
...,...,...,...,...,...,...,...,...
CO2 Emissions Consumed,71017.0,32058.196950,9.277677e+03,-1506.393360,25123.548578,30180.900185,37375.739667,7.221896e+04
Positive Generation,71173.0,95406.649305,1.648786e+04,287.000000,83762.000000,92449.000000,104546.000000,1.642810e+05
Consumed Electricity,71020.0,91895.396860,1.673486e+04,-8442.000000,80066.750000,89106.000000,101299.500000,1.635450e+05
CO2 Emissions Intensity for Generated Electricity,71173.0,0.755716,9.855004e-02,0.062798,0.680945,0.748872,0.824168,1.080516e+00


In [ ]:
#understanding data ranges in our data
print("Start:", hourly_df["UTC time"].min())
print("End:", hourly_df["UTC time"].max())

Start: 2015-07-01 05:00:00
End: 2026-11-02 05:00:00


## 3. Cleaning and preprocessing

In [10]:
hourly = pd.read_excel(
    excel_file,
    sheet_name="Published Hourly Data"
)

hourly["UTC time"] = pd.to_datetime(hourly["UTC time"])

print("Rows and columns:", hourly.shape)
print("Earliest timestamp:", hourly["UTC time"].min())
print("Latest timestamp:", hourly["UTC time"].max())

Rows and columns: (99409, 108)
Earliest timestamp: 2015-07-01 05:00:00
Latest timestamp: 2026-11-02 05:00:00


### Creating separate subset for our study

In [11]:
local_dates = pd.to_datetime(hourly["Local date"])

pjm = hourly.loc[
    (local_dates >= "2022-01-01") &
    (local_dates < "2026-01-01")
].copy()

print("Study-period rows and columns:", pjm.shape)
print("First local date:", pjm["Local date"].min())
print("Last local date:", pjm["Local date"].max())

Study-period rows and columns: (35064, 108)
First local date: 2022-01-01 00:00:00
Last local date: 2025-12-31 00:00:00


In [12]:
for column in pjm.columns:
    print(column)

BA
UTC time
Local date
Hour
Local time
Time zone
Generation only?
Demand forecast
Demand
Net generation
Total interchange
Imputed demand
Imputed net generation
Imputed total interchange
Adjusted demand
Adjusted net generation
Adjusted total interchange
NG: COL
NG: NG
NG: NUC
NG: OIL
NG: GEO
NG: WAT
NG: PS
NG: SUN
NG: SNB
NG: WND
NG: WNB
NG: BAT
NG: OES
NG: UES
NG: OTH
NG: UNK
Imputed COL Gen
Imputed NG Gen
Imputed NUC Gen
Imputed OIL Gen
Imputed GEO Gen
Imputed WAT Gen
Imputed PS Gen
Imputed SUN Gen
Imputed SNB Gen
Imputed WND Gen
Imputed WNB Gen
Imputed BAT Gen
Imputed OES Gen
Imputed UES Gen
Imputed OTH Gen
Imputed UNK Gen
Adjusted COL Gen
Adjusted NG Gen
Adjusted NUC Gen
Adjusted OIL Gen
Adjusted GEO Gen
Adjusted WAT Gen
Adjusted PS Gen
Adjusted SUN Gen
Adjusted SNB Gen
Adjusted WND Gen
Adjusted WNB Gen
Adjusted BAT Gen
Adjusted OES Gen
Adjusted UES Gen
Adjusted OTH Gen
Adjusted UNK Gen
CPLE
CPLW
DUK
LGEE
MISO
NYIS
OVEC
TVA
Subregion AP
Subregion AEP
Subregion ATSI
Subregion AE
Subr